# RAID v8: Sentence-by-Sentence Context Reveal with Shuffled Control

**Method:** Reveal context one SENTENCE at a time (not one token). Measure perplexity on target sentence with 0, 1, 2, ... N prior sentences. Run on both intact and sentence-shuffled versions of each document.

**Why this is better than v7:**
- Context always starts/ends at sentence boundaries — no fragment noise
- Shuffled control uses sentence-order shuffling (not token shuffling) — each increment adds a complete sentence
- Measures influence in sentence distance — the natural unit for discourse coherence
- Subtraction is clean because both conditions add comparable units (whole sentences)

**Produces:** Power law exponent for coherence decay measured in sentences, with proper shuffled correction. Human vs AI comparison.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import uniform_filter1d
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

In [ ]:
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/raid_sampled")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/RAID_v8")
    if (DRIVE_DATA / "raid_corpus.jsonl").exists():
        DATA_DIR = DRIVE_DATA
    else:
        LOCAL_DATA = Path("/content/data/raid_sampled")
        if not (LOCAL_DATA / "raid_corpus.jsonl").exists():
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/raid_v8")
    DATA_DIR = Path("../data/raid_sampled")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True
N_DOCS = 150
RANDOM_SEED = 42
MIN_SENTENCES = 8  # need enough sentences for a meaningful curve
DOMAINS = ['abstracts', 'books', 'news', 'poetry', 'recipes', 'reddit', 'reviews', 'wiki']

print(f"N docs per population: {N_DOCS}")

In [ ]:
corpus_all = []
with open(DATA_DIR / "raid_corpus.jsonl") as f:
    for line in f:
        corpus_all.append(json.loads(line))

def split_sentences(text):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sents if len(s.strip().split()) >= 4]

rng = np.random.RandomState(RANDOM_SEED)
sample = []
for pop in ['human', 'ai']:
    pop_docs = [d for d in corpus_all if d['population'] == pop]
    # Filter to docs with enough sentences
    pop_docs = [d for d in pop_docs if len(split_sentences(d['text'])) >= MIN_SENTENCES]
    for domain in DOMAINS:
        pool = [d for d in pop_docs if d['domain'] == domain]
        n = min(len(pool), N_DOCS // len(DOMAINS) + 1)
        if n > 0:
            sample.extend(rng.choice(pool, size=n, replace=False))

print(f"Selected {len(sample)} documents")
print(f"  Human: {sum(1 for d in sample if d['population'] == 'human')}")
print(f"  AI: {sum(1 for d in sample if d['population'] == 'ai')}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("Model loaded")

In [ ]:
@torch.no_grad()
def compute_ppl(token_ids, target_start, target_end):
    if target_start >= target_end - 1:
        return float('inf')
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[token_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')


def compute_sentence_reveal_curve(doc, shuffled=False, rng_shuf=None):
    sentences = split_sentences(doc['text'])
    if len(sentences) < MIN_SENTENCES:
        return None

    # Tokenize each sentence
    sent_ids = [tokenizer.encode(s, add_special_tokens=False) for s in sentences]

    # Target = sentence at ~65% through the document
    tgt_idx = int(len(sentences) * 0.65)
    tgt_idx = max(3, min(tgt_idx, len(sentences) - 1))
    target_ids = sent_ids[tgt_idx]
    if len(target_ids) < 3:
        return None

    # Prior sentences (context pool)
    prior_sents = list(range(tgt_idx))  # indices 0 to tgt_idx-1

    if shuffled and rng_shuf is not None:
        rng_shuf.shuffle(prior_sents)

    # Reveal context one sentence at a time
    # 0 sentences = just target alone
    # 1 sentence = most recent prior sentence + target
    # 2 sentences = two most recent + target
    # etc.
    ppls = []
    n_context_sents = []

    # 0 context
    ppl = compute_ppl(target_ids, 0, len(target_ids))
    if not math.isinf(ppl):
        ppls.append(ppl)
        n_context_sents.append(0)

    # Add sentences one at a time (most recent first for intact, random order for shuffled)
    if shuffled:
        order = prior_sents  # already shuffled
    else:
        order = list(reversed(prior_sents))  # most recent first

    context_ids = []
    for k, sent_idx in enumerate(order):
        # Prepend this sentence to context
        context_ids = sent_ids[sent_idx] + context_ids
        full = context_ids + target_ids
        ppl = compute_ppl(full, len(context_ids), len(full))
        if not math.isinf(ppl):
            ppls.append(ppl)
            n_context_sents.append(k + 1)

    if len(ppls) < 4:
        return None

    return {
        'doc_id': doc['doc_id'],
        'domain': doc['domain'],
        'population': doc['population'],
        'n_total_sents': len(sentences),
        'target_idx': tgt_idx,
        'n_context_sents': n_context_sents,
        'ppls': ppls,
    }


print("Functions defined")

In [ ]:
results_path = BASE_DIR / "v8_intact.json"
shuffled_path = BASE_DIR / "v8_shuffled.json"

if results_path.exists() and shuffled_path.exists():
    with open(results_path) as f:
        intact_curves = json.load(f)
    with open(shuffled_path) as f:
        shuffled_curves = json.load(f)
    print(f"Loaded {len(intact_curves)} intact + {len(shuffled_curves)} shuffled")
else:
    intact_curves = []
    shuffled_curves = []
    rng_s = np.random.RandomState(RANDOM_SEED + 77)

    for doc in tqdm(sample, desc="Sentence-by-sentence reveal"):
        # Intact
        result = compute_sentence_reveal_curve(doc, shuffled=False)
        if result is not None:
            intact_curves.append(result)
            # Shuffled version of same doc
            result_s = compute_sentence_reveal_curve(doc, shuffled=True, rng_shuf=rng_s)
            if result_s is not None:
                shuffled_curves.append(result_s)

    with open(results_path, 'w') as f:
        json.dump(intact_curves, f)
    with open(shuffled_path, 'w') as f:
        json.dump(shuffled_curves, f)
    print(f"Computed {len(intact_curves)} intact + {len(shuffled_curves)} shuffled")

print(f"Intact — Human: {sum(1 for c in intact_curves if c['population']=='human')}, AI: {sum(1 for c in intact_curves if c['population']=='ai')}")
print(f"Shuffled — Human: {sum(1 for c in shuffled_curves if c['population']=='human')}, AI: {sum(1 for c in shuffled_curves if c['population']=='ai')}")

In [ ]:
# Compute mean curves by population
max_sents = 20  # max context sentences to plot

def compute_mean_ppl_by_n_sents(curves, max_n=20):
    by_n = {n: [] for n in range(max_n + 1)}
    for curve in curves:
        for n, ppl in zip(curve['n_context_sents'], curve['ppls']):
            if n <= max_n:
                by_n[n].append(ppl)
    ns = sorted([n for n in by_n if len(by_n[n]) >= 5])
    means = [np.mean(by_n[n]) for n in ns]
    sems = [np.std(by_n[n]) / np.sqrt(len(by_n[n])) for n in ns]
    return ns, means, sems

def compute_normalized(ns, means):
    means = np.array(means)
    if means[0] - means[-1] > 0:
        return (means[0] - means) / (means[0] - means[-1])
    return np.zeros_like(means)

h_intact = [c for c in intact_curves if c['population'] == 'human']
a_intact = [c for c in intact_curves if c['population'] == 'ai']
h_shuf = [c for c in shuffled_curves if c['population'] == 'human']
a_shuf = [c for c in shuffled_curves if c['population'] == 'ai']

h_i_ns, h_i_means, h_i_sems = compute_mean_ppl_by_n_sents(h_intact, max_sents)
a_i_ns, a_i_means, a_i_sems = compute_mean_ppl_by_n_sents(a_intact, max_sents)
h_s_ns, h_s_means, h_s_sems = compute_mean_ppl_by_n_sents(h_shuf, max_sents)
a_s_ns, a_s_means, a_s_sems = compute_mean_ppl_by_n_sents(a_shuf, max_sents)

fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# A: Raw perplexity — intact vs shuffled
ax = axes[0, 0]
ax.errorbar(h_i_ns, h_i_means, yerr=h_i_sems, fmt='o-', color='blue', linewidth=2, markersize=4, capsize=2, label='Human intact')
ax.errorbar(h_s_ns, h_s_means, yerr=h_s_sems, fmt='s:', color='blue', linewidth=1.5, markersize=4, capsize=2, label='Human shuffled', alpha=0.6)
ax.errorbar(a_i_ns, a_i_means, yerr=a_i_sems, fmt='o-', color='red', linewidth=2, markersize=4, capsize=2, label='AI intact')
ax.errorbar(a_s_ns, a_s_means, yerr=a_s_sems, fmt='s:', color='red', linewidth=1.5, markersize=4, capsize=2, label='AI shuffled', alpha=0.6)
ax.set_xlabel('Number of Context Sentences')
ax.set_ylabel('Perplexity')
ax.set_title('A. Raw Perplexity: Intact vs Shuffled', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# B: Normalized curves
ax = axes[0, 1]
for ns, means, color, ls, label in [
    (h_i_ns, h_i_means, 'blue', '-', 'Human intact'),
    (h_s_ns, h_s_means, 'blue', ':', 'Human shuffled'),
    (a_i_ns, a_i_means, 'red', '-', 'AI intact'),
    (a_s_ns, a_s_means, 'red', ':', 'AI shuffled'),
]:
    norm = compute_normalized(ns, means)
    ax.plot(ns, norm, f'o{ls}', color=color, linewidth=2, markersize=4, label=label, alpha=0.8 if ':' not in ls else 0.5)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Number of Context Sentences')
ax.set_ylabel('Fraction of Total Benefit')
ax.set_title('B. Normalized Curves', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# C: Marginal gain per sentence
ax = axes[0, 2]
for ns, means, color, ls, label in [
    (h_i_ns, h_i_means, 'blue', '-', 'Human intact'),
    (h_s_ns, h_s_means, 'blue', ':', 'Human shuffled'),
    (a_i_ns, a_i_means, 'red', '-', 'AI intact'),
    (a_s_ns, a_s_means, 'red', ':', 'AI shuffled'),
]:
    marg = [-1 * (means[i] - means[i-1]) for i in range(1, len(means))]
    ax.plot(ns[1:], marg, f'o{ls}', color=color, linewidth=1.5, markersize=4, label=label, alpha=0.8 if ':' not in ls else 0.5)
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Number of Context Sentences')
ax.set_ylabel('Marginal PPL Drop per Sentence')
ax.set_title('C. Marginal Gain per Sentence', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# D: Corrected marginals (intact - shuffled)
ax = axes[1, 0]
# Align on common sentence counts
common_ns = sorted(set(h_i_ns) & set(h_s_ns))
h_i_dict = dict(zip(h_i_ns, h_i_means))
h_s_dict = dict(zip(h_s_ns, h_s_means))
a_i_dict = dict(zip(a_i_ns, a_i_means))
a_s_dict = dict(zip(a_s_ns, a_s_means))

for pop_i, pop_s, color, label in [
    (h_i_dict, h_s_dict, 'blue', 'Human'),
    (a_i_dict, a_s_dict, 'red', 'AI'),
]:
    common = sorted(set(pop_i.keys()) & set(pop_s.keys()))
    if len(common) >= 4:
        i_vals = [pop_i[n] for n in common]
        s_vals = [pop_s[n] for n in common]
        corr_marg = [-1*((i_vals[j] - i_vals[j-1]) - (s_vals[j] - s_vals[j-1])) for j in range(1, len(common))]
        ax.plot(common[1:], corr_marg, 'o-', color=color, linewidth=2, markersize=5, label=label)
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Number of Context Sentences')
ax.set_ylabel('Corrected Marginal (intact - shuffled)')
ax.set_title('D. Pure Coherence Signal per Sentence', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# E: Power law fit on corrected marginals
ax = axes[1, 1]
for pop_i, pop_s, color, label in [
    (h_i_dict, h_s_dict, 'blue', 'Human'),
    (a_i_dict, a_s_dict, 'red', 'AI'),
]:
    common = sorted(set(pop_i.keys()) & set(pop_s.keys()))
    if len(common) >= 4:
        i_vals = [pop_i[n] for n in common]
        s_vals = [pop_s[n] for n in common]
        corr_marg = [-1*((i_vals[j] - i_vals[j-1]) - (s_vals[j] - s_vals[j-1])) for j in range(1, len(common))]
        x = np.array(common[1:])
        y = np.array(corr_marg)
        pos = y > 0
        if sum(pos) >= 3:
            slope, intercept, r, p, _ = stats.linregress(np.log(x[pos]), np.log(y[pos]))
            ax.plot(x, y, 'o', color=color, markersize=6)
            fit_x = np.linspace(x[pos].min(), x[pos].max(), 50)
            ax.plot(fit_x, np.exp(intercept) * fit_x**slope, '--', color=color,
                    label=f'{label}: d^{slope:.2f} (r={r:.2f})')
            print(f"CORRECTED {label}: exponent={slope:.3f}, r={r:.3f}, p={p:.4f}")
ax.set_xscale('log')
ax.set_xlabel('Sentence Distance (log)')
ax.set_ylabel('Corrected Marginal')
ax.set_title('E. Power Law (Sentence Distance)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# F: Genre comparison (human intact only, normalized)
ax = axes[1, 2]
genre_colors = dict(zip(DOMAINS, plt.cm.Set2.colors[:len(DOMAINS)]))
for domain in DOMAINS:
    domain_curves = [c for c in h_intact if c['domain'] == domain]
    if len(domain_curves) >= 3:
        ns, means, _ = compute_mean_ppl_by_n_sents(domain_curves, max_sents)
        if len(ns) >= 4:
            norm = compute_normalized(ns, means)
            ax.plot(ns, norm, 'o-', color=genre_colors[domain], linewidth=1.5, markersize=3, label=domain)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Number of Context Sentences')
ax.set_ylabel('Fraction of Total Benefit')
ax.set_title('F. Genre Comparison (Human)', fontweight='bold')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.2)

plt.suptitle('Sentence-by-Sentence Context Reveal: Corrected Power Law',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_v8_sentence_reveal.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print("\nSUMMARY:")
print(f"  Intact curves: {len(intact_curves)} ({sum(1 for c in intact_curves if c['population']=='human')} human, {sum(1 for c in intact_curves if c['population']=='ai')} AI)")
print(f"  Shuffled curves: {len(shuffled_curves)}")